# SciPy Computation Library EDA

SciPy is useful for statistical features, signal processing, optimization, and target diagnostics. This notebook shows concrete feature-engineering and target-engineering uses.

In [1]:
from __future__ import annotations

import importlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

def required_library_status(module_name: str) -> pd.DataFrame:
    try:
        module = importlib.import_module(module_name)
        return pd.DataFrame([{
            "module": module_name,
            "required": True,
            "installed": True,
            "version": getattr(module, "__version__", "unknown"),
            "module_file": getattr(module, "__file__", None),
        }])
    except Exception as exc:
        return pd.DataFrame([{
            "module": module_name,
            "required": True,
            "installed": False,
            "version": None,
            "module_file": None,
            "error": f"{type(exc).__name__}: {exc}",
        }])

def synthetic_prices(rows: int = 360, symbol: str = "AAPL") -> pd.DataFrame:
    rng = np.random.default_rng(7)
    dates = pd.date_range("2024-01-02", periods=rows, freq="B")
    drift = 0.00055
    shock = rng.normal(0, 0.015, rows)
    close = 100 * np.exp(np.cumsum(drift + shock))
    open_ = close * (1 + rng.normal(0, 0.003, rows))
    high = np.maximum(open_, close) * (1 + rng.uniform(0.001, 0.02, rows))
    low = np.minimum(open_, close) * (1 - rng.uniform(0.001, 0.02, rows))
    volume = rng.integers(900_000, 4_500_000, rows).astype(float)
    return pd.DataFrame({
        "symbol": symbol,
        "date": dates,
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "volume": volume,
    })

prices = synthetic_prices()
prices.head()

,symbol,date,open,high,low,close,volume
0,AAPL,2024-01-02,100.2080,101.2355,98.8738,100.0569,"4,479,129.0000"
1,AAPL,2024-01-03,101.1259,102.2340,100.4497,100.5615,"2,365,734.0000"
2,AAPL,2024-01-04,100.3819,101.9042,99.3952,100.2040,"1,409,205.0000"
3,AAPL,2024-01-05,98.9452,99.8670,97.0239,98.9286,"4,225,356.0000"
4,AAPL,2024-01-08,97.8130,98.6722,97.4943,98.3103,"2,886,674.0000"


## Required Dependency Status

In [2]:
required_library_status("scipy")

,module,required,installed,version,module_file
0,scipy,True,True,1.18.0,/home/jlee153232/miniconda3/lib/python3.13/sit...


## Feature Engineering: Statistical And Signal Features

In [3]:
from scipy import signal, stats

work = prices.set_index("date")
returns = work["close"].pct_change()
scipy_features = pd.DataFrame(index=work.index)
scipy_features["sp__zscore_ret_20"] = returns.rolling(20).apply(lambda x: stats.zscore(x, nan_policy="omit")[-1] if np.isfinite(x).sum() >= 5 else np.nan, raw=False)
scipy_features["sp__skew_ret_63"] = returns.rolling(63).apply(lambda x: stats.skew(x, nan_policy="omit"), raw=False)
scipy_features["sp__kurt_ret_63"] = returns.rolling(63).apply(lambda x: stats.kurtosis(x, nan_policy="omit"), raw=False)
# Savitzky-Golay smoothing is causal only if shifted; use the smoothed value as a diagnostic and shift for no-lookahead features.
smoothed = pd.Series(signal.savgol_filter(work["close"], window_length=21, polyorder=2), index=work.index)
scipy_features["sp__savgol_trend_21_lag1"] = smoothed.shift(1)
scipy_features["sp__savgol_dist_21_lag1"] = work["close"] / scipy_features["sp__savgol_trend_21_lag1"] - 1
scipy_features.tail(8)

,sp__zscore_ret_20,sp__skew_ret_63,sp__kurt_ret_63,sp__savgol_trend_21_lag1,sp__savgol_dist_21_lag1
date,,,,,
2025-05-08,0.1823,0.2610,-0.5733,70.0979,-0.0077
2025-05-09,-0.2864,0.3601,-0.5940,70.0838,-0.0100
2025-05-12,0.0614,0.3795,-0.4659,70.0502,-0.0058
2025-05-13,0.0784,0.4088,-0.3801,69.9971,-0.0001
2025-05-14,-0.4997,0.4688,-0.2834,69.9246,-0.0030
2025-05-15,-0.2672,0.5091,-0.2141,69.8326,-0.0017
2025-05-16,0.0745,0.5033,-0.2206,69.7212,0.0035
2025-05-19,-0.1918,0.5204,-0.2071,69.5904,0.0047


## Target Engineering: Distribution Diagnostics

In [4]:
from quant_warehouse.platforms.data_providers.fmp.target_engineering import build_forward_return_labels
from scipy import stats

labels = build_forward_return_labels(prices[["symbol", "date", "close"]], horizons=[5, 20, 60])
diagnostics = []
for horizon, group in labels.dropna(subset=["target_value"]).groupby("horizon"):
    values = group["target_value"].to_numpy()
    diagnostics.append({
        "horizon": horizon,
        "n": len(values),
        "mean": values.mean(),
        "std": values.std(),
        "skew": stats.skew(values),
        "kurtosis": stats.kurtosis(values),
        "normaltest_pvalue": stats.normaltest(values).pvalue if len(values) >= 8 else np.nan,
    })
pd.DataFrame(diagnostics)

,horizon,n,mean,std,skew,kurtosis,normaltest_pvalue
0,5,355,-0.0045,0.0306,-0.3027,0.7341,0.0040
1,20,340,-0.0174,0.0671,-0.0214,-0.2081,0.7699
2,60,300,-0.0459,0.0911,0.1803,-0.8559,0.0000


## Target Engineering: Mean-Variance Style Optimization

In [5]:
from scipy.optimize import minimize

assets = ["AAA", "BBB", "CCC"]
mu = np.array([0.08, 0.12, 0.05])
cov = np.array([[0.040, 0.012, 0.006], [0.012, 0.090, 0.010], [0.006, 0.010, 0.025]])
risk_aversion = 2.5

def objective(w):
    return -(mu @ w - risk_aversion * (w @ cov @ w))

constraints = ({"type": "eq", "fun": lambda w: w.sum() - 1.0},)
bounds = [(0.0, 0.7)] * len(assets)
result = minimize(objective, x0=np.ones(len(assets))/len(assets), bounds=bounds, constraints=constraints, method="SLSQP")
pd.DataFrame({"asset": assets, "weight": result.x, "expected_return": mu})

,asset,weight,expected_return
0,AAA,0.3844,0.0800
1,BBB,0.2203,0.1200
2,CCC,0.3953,0.0500


## Notes

- SciPy should power statistical diagnostics and optimization helpers, not vendor-specific data fetching.
- Watch for lookahead in filters: centered smoothing must be shifted or replaced with causal filters.
- Useful promotion candidates: regime z-scores, robust outlier filters, distribution tests for labels, and optimization diagnostics for option/portfolio targets.